# Historical experimental notebook

Read-only execution history, **not Run All safe**. Cells were edited and rerun interactively. Package installation, recovery and repeated wrapping cells must not be replayed as a sequence. Use `reproduce.ipynb` for the cleaned workflow.

Original code retained; widget/HTML outputs and machine metadata removed. Plain-text results are historical observations.

**Task Definition**
## What the model receives
- The model receives a current household preference and an instruction
## What it returns
- It only returns the updated changes. If no updates are made to the state return {
  "updates": {},
  "clarify": []
}

## The fields it supports
| Field        | Allowed values                              |
| ------------ | ------------------------------------------- |
| `adults`     | Whole number from 1 to 20                   |
| `budget_eur` | Number from 1 to 10,000, expressed in euros |
| `dairy`      | `"dairy_free"` or `"no_restriction"`        |

## When it should request clarification
When an instruction is ambigous it should ask for clarity.

## Examples
- Only two adults this week. Keep the budget at 60 euros. Output should be:
{
  "updates": {
    "adults": 2,
    "budget_eur": 60
  },
  "clarify": []
}
- “Set the budget to 50 euros; we might have three or five adults". It should Apply the clear part; clarify the uncertain part
{"updates":{"budget_eur":50},"clarify":["adults"]}

In [1]:
%pip install -q -U transformers peft datasets accelerate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 42.2 MB/s eta 0:00:00


In [5]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [2]:
import json, pathlib, subprocess, sys, torch
from google.colab import files

assert torch.cuda.is_available(), "Select a GPU runtime before training."
print(torch.cuda.get_device_name(0))
ROOT = pathlib.Path('/content/household-lora')
ROOT.mkdir(exist_ok=True)
# Reuse the files from the upload that just completed.
# Accept both "train.jsonl" and names such as "train (2).jsonl".
import re

selected = {}

for name in ("train.jsonl", "val.jsonl", "test.jsonl"):
    stem = name.removesuffix(".jsonl")
    pattern = rf"{re.escape(stem)}(?: \(\d+\))?\.jsonl"

    matches = [
        key for key in uploaded
        if re.fullmatch(pattern, key)
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected one uploaded file for {name}; "
            f"found {matches}. Uploaded names: {list(uploaded)}"
        )

    selected[name] = matches[0]

for name, uploaded_name in selected.items():
    (ROOT / name).write_bytes(uploaded[uploaded_name])
    print(f"Loaded {uploaded_name} as {name}")

FIELDS = {'adults', 'budget_eur', 'dairy'}
def schema_ok(o):
    if not isinstance(o, dict) or set(o) != {'updates', 'clarify'}:
        return False
    u, c = o['updates'], o['clarify']
    if not isinstance(u, dict) or not set(u) <= FIELDS:
        return False
    if not isinstance(c, list) or not all(isinstance(x, str) for x in c):
        return False
    if not set(c) <= FIELDS or len(c) != len(set(c)) or set(u) & set(c):
        return False
    if 'adults' in u and (type(u['adults']) is not int or not 1 <= u['adults'] <= 20):
        return False
    if 'budget_eur' in u and (type(u['budget_eur']) not in (int, float)
                              or not 1 <= u['budget_eur'] <= 10000):
        return False
    if 'dairy' in u and u['dairy'] not in ('dairy_free', 'no_restriction'):
        return False
    return True

def read_split(name):
    rows = [json.loads(s) for s in (ROOT / f'{name}.jsonl').read_text().splitlines() if s.strip()]
    assert rows
    for r in rows:
        assert set(r) == {'id', 'family', 'input', 'target'}, r
        assert set(r['input']) == {'current_state', 'instruction'}, r
        assert isinstance(r['input']['instruction'], str) and r['input']['instruction'].strip()
        assert isinstance(r['family'], str) and r['family']
        assert set(r['input']['current_state']) == FIELDS
        assert schema_ok({'updates': r['input']['current_state'], 'clarify': []}), r
        assert schema_ok(r['target']), r
    return rows

data = {s: read_split(s) for s in ('train', 'val', 'test')}
all_rows = sum(data.values(), [])
assert len({r['id'] for r in all_rows}) == len(all_rows)
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    assert not ({r['family'] for r in data[a]} & {r['family'] for r in data[b]})
    canonical = lambda r: json.dumps(r['input'], sort_keys=True)
    assert not ({canonical(r) for r in data[a]} & {canonical(r) for r in data[b]})
print({s: len(rows) for s, rows in data.items()})

Tesla T4
Loaded train (2).jsonl as train.jsonl
Loaded val (2).jsonl as val.jsonl
Loaded test (2).jsonl as test.jsonl
{'train': 80, 'val': 16, 'test': 24}


In [3]:
from huggingface_hub import model_info
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

set_seed(42)
MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
REVISION = json.loads(
    (ROOT / "recovery.json").read_text()
)["base_revision"]
tok = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
tok.padding_side = 'right'
model = AutoModelForCausalLM.from_pretrained(
    MODEL, revision=REVISION, torch_dtype=torch.float32
).to('cuda')

SYSTEM = '''Extract household preference updates. Return ONLY a JSON object with
exactly keys "updates" (object) and "clarify" (list of field names).
Allowed fields: adults (integer 1..20), budget_eur (number 1..10000),
dairy ("dairy_free" or "no_restriction").
Include explicit requested values in updates even if already equal to current state.
Omit fields with no requested change. Do not infer dairy restrictions from product names.
Resolve explicit self-corrections using the corrected value.
For ambiguity, unresolved contradiction, out-of-range values, relative numerical
changes or non-euro budgets, omit that field from updates and include it in clarify.
Do not update and clarify the same field. Ignore unsupported fields.
Use an empty object and empty list when no changes or clarifications are needed.'''

def prompt_for(r):
    return tok.apply_chat_template([
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': json.dumps(r['input'], ensure_ascii=False)}
    ], tokenize=False, add_generation_prompt=True)

def predict(r):
    model.eval()
    inputs = tok(prompt_for(r), add_special_tokens=False, return_tensors='pt').to('cuda')
    with torch.inference_mode():
        output = model.generate(**inputs, do_sample=False, max_new_tokens=160,
                                pad_token_id=tok.pad_token_id,
                                eos_token_id=tok.eos_token_id)
    return tok.decode(output[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def save_predictions(split, tag):
    rows = [{'id': r['id'], 'raw': predict(r)} for r in data[split]]
    (ROOT / f'{tag}-{split}.json').write_text(json.dumps(rows, indent=2))
    return rows

(ROOT / 'prompt.txt').write_text(SYSTEM)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

801

In [4]:
# Restore the revised prompt, replacing Cell 3's original SYSTEM text.
SYSTEM = (ROOT / "prompt.txt").read_text()

# Restore baseline predictions without printing test outputs.
base_val = json.loads((ROOT / "base-val.json").read_text())
base_test = json.loads((ROOT / "base-test.json").read_text())

# Confirm that the optional conflicting package is gone.
import importlib.util
assert importlib.util.find_spec("torchao") is None

print("Revised prompt and baseline restored. Run Cell 5, then Cell 6.")

Revised prompt and baseline restored. Run Cell 5, then Cell 6.


In [4]:
base_val = save_predictions('val', 'base')
for r, p in zip(data['val'][:4], base_val[:4]):
    print(r['input'], '\nEXPECTED:', r['target'], '\nRAW:', p['raw'])
# Fix any contract defect now using validation, then rerun before freezing.
base_test = save_predictions('test', 'base')  # save; don't tune against these

{'current_state': {'adults': 3, 'budget_eur': 75, 'dairy': 'dairy_free'}, 'instruction': "Let's move the budget down to 55 euros."} 
EXPECTED: {'updates': {'budget_eur': 55}, 'clarify': []} 
RAW: ```json
{
  "updates": {},
  "clarify": [
    "budget_eur"
  ]
}
```
{'current_state': {'adults': 2, 'budget_eur': 60, 'dairy': 'no_restriction'}, 'instruction': 'We need to go dairy-free starting this week.'} 
EXPECTED: {'updates': {'dairy': 'dairy_free'}, 'clarify': []} 
RAW: ```json
{
  "updates": {},
  "clarify": [
    "dairy"
  ]
}
```
{'current_state': {'adults': 2, 'budget_eur': 70, 'dairy': 'no_restriction'}, 'instruction': "From now on it's 5 adults and a 140 euro budget."} 
EXPECTED: {'updates': {'adults': 5, 'budget_eur': 140}, 'clarify': []} 
RAW: ```json
{
  "updates": {},
  "clarify": [
    "budget_eur",
    "dairy"
  ]
}
```
{'current_state': {'adults': 6, 'budget_eur': 300, 'dairy': 'dairy_free'}, 'instruction': "We're 3 adults now, no dairy restriction, and 90 euros total."} 


In [5]:
import shutil

(ROOT / "prompt-v1.txt").write_text(SYSTEM)

for split in ("val", "test"):
    source = ROOT / f"base-{split}.json"
    if source.exists():
        shutil.copy2(source, ROOT / f"base-v1-{split}.json")

In [5]:
SYSTEM = """
You extract explicit household preference updates from a user instruction.
The input contains current_state and instruction.

Return exactly one JSON object:
{"updates": {}, "clarify": []}

OUTPUT RULES:
- Return JSON only. No Markdown fences, explanation, or extra text.
- Always include both "updates" and "clarify".
- "updates" contains only fields the user explicitly requests to set.
- "clarify" contains only field names whose requested values are unclear
  or unsupported by the rules below.
- Never update and clarify the same field.
- Never clarify a field just because the user did not mention it.

ALLOWED FIELDS:
- adults: integer from 1 to 20.
- budget_eur: number from 1 to 10000, expressed in euros.
- dairy: "dairy_free" or "no_restriction".

INTERPRETATION RULES:
- When a final value is explicit and valid, put it in updates.
- A request to increase or decrease something TO a stated value
  gives an explicit final value. Apply that value.
- A request to increase or decrease something BY an amount is a
  relative numerical change. For this task, clarify that field.
- An explicit dairy-free preference means "dairy_free".
- Removing the dairy restriction means "no_restriction".
- A product request alone does not establish a dairy preference.
- An explicit self-correction replaces the earlier value.
- Unresolved alternatives, contradictions, out-of-range values,
  and non-euro budgets require clarification of the affected field.
- If an explicit requested value equals the current value,
  still include it in updates.
- Instructions to keep a field unchanged do not produce an update.
- Ignore unsupported fields and unrelated content.
- If nothing needs updating or clarification, return:
  {"updates": {}, "clarify": []}
""".strip()

In [7]:
revised_val = save_predictions("val", "prompt-v2")

for row, prediction in zip(data["val"], revised_val):
    print("INPUT:", row["input"])
    print("EXPECTED:", row["target"])
    print("RAW:", prediction["raw"])
    print()

INPUT: {'current_state': {'adults': 3, 'budget_eur': 75, 'dairy': 'dairy_free'}, 'instruction': "Let's move the budget down to 55 euros."}
EXPECTED: {'updates': {'budget_eur': 55}, 'clarify': []}
RAW: {"updates": {}, "clarify": ["set budget to 55 euros"]}

INPUT: {'current_state': {'adults': 2, 'budget_eur': 60, 'dairy': 'no_restriction'}, 'instruction': 'We need to go dairy-free starting this week.'}
EXPECTED: {'updates': {'dairy': 'dairy_free'}, 'clarify': []}
RAW: {"updates": {}, "clarify": ["dairy_free"]}

INPUT: {'current_state': {'adults': 2, 'budget_eur': 70, 'dairy': 'no_restriction'}, 'instruction': "From now on it's 5 adults and a 140 euro budget."}
EXPECTED: {'updates': {'adults': 5, 'budget_eur': 140}, 'clarify': []}
RAW: {"updates": {}, "clarify": ["budget_eur"]}

INPUT: {'current_state': {'adults': 6, 'budget_eur': 300, 'dairy': 'dairy_free'}, 'instruction': "We're 3 adults now, no dairy restriction, and 90 euros total."}
EXPECTED: {'updates': {'adults': 3, 'dairy': 'no_r

In [8]:
# Save the revised prompt that produced the validation outputs above.
(ROOT / "prompt.txt").write_text(SYSTEM)

# Make the revised validation predictions our official baseline.
base_val = json.loads(
    (ROOT / "prompt-v2-val.json").read_text()
)

(ROOT / "base-val.json").write_text(
    json.dumps(base_val, indent=2)
)

# Regenerate test predictions with the SAME revised prompt.
# Save them without displaying or evaluating them.
base_test = save_predictions("test", "base")

print("Prompt and baseline predictions saved.")
print("Validation examples:", len(base_val))
print("Test examples saved without inspection:", len(base_test))

Prompt and baseline predictions saved.
Validation examples: 16
Test examples saved without inspection: 24


In [6]:
from datasets import Dataset

def encode(r):
    p = tok(prompt_for(r), add_special_tokens=False)['input_ids']
    answer = json.dumps(r['target'], ensure_ascii=False, separators=(',', ':')) + tok.eos_token
    a = tok(answer, add_special_tokens=False)['input_ids']
    ids = p + a
    assert len(ids) <= 512, (r['id'], len(ids))
    assert a and a[-1] == tok.eos_token_id
    return {'input_ids': ids, 'attention_mask': [1] * len(ids),
            'labels': [-100] * len(p) + a}

train_ds = Dataset.from_list([encode(r) for r in data['train']])
val_ds = Dataset.from_list([encode(r) for r in data['val']])
sample = train_ds[0]
print('FULL:', tok.decode(sample['input_ids']))
print('LOSS TARGET:', tok.decode([x for x in sample['labels'] if x != -100]))

def collate(rows):
    width = max(len(r['input_ids']) for r in rows)
    return {key: torch.tensor([
        r[key] + [pad] * (width - len(r[key])) for r in rows
    ]) for key, pad in [('input_ids', tok.pad_token_id), ('attention_mask', 0), ('labels', -100)]}

assert any(x == -100 for x in sample['labels'])
assert any(x != -100 for x in sample['labels'])

FULL: <|im_start|>system
You extract explicit household preference updates from a user instruction.
The input contains current_state and instruction.

Return exactly one JSON object:
{"updates": {}, "clarify": []}

OUTPUT RULES:
- Return JSON only. No Markdown fences, explanation, or extra text.
- Always include both "updates" and "clarify".
- "updates" contains only fields the user explicitly requests to set.
- "clarify" contains only field names whose requested values are unclear
  or unsupported by the rules below.
- Never update and clarify the same field.
- Never clarify a field just because the user did not mention it.

ALLOWED FIELDS:
- adults: integer from 1 to 20.
- budget_eur: number from 1 to 10000, expressed in euros.
- dairy: "dairy_free" or "no_restriction".

INTERPRETATION RULES:
- When a final value is explicit and valid, put it in updates.
- A request to increase or decrease something TO a stated value
  gives an explicit final value. Apply that value.
- A request to i

In [8]:
import time
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments

model = get_peft_model(model, LoraConfig(
    task_type='CAUSAL_LM', r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'], bias='none'
))
model.print_trainable_parameters()
trainable = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
assert trainable and all('lora_' in n for n, _ in trainable)
model.config.use_cache = False
bf16 = torch.cuda.is_bf16_supported()
args = TrainingArguments(
    output_dir=str(ROOT / 'checkpoints'),
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=8, num_train_epochs=2, learning_rate=1e-4,
    warmup_steps=0.1, logging_steps=1, eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='eval_loss',
    greater_is_better=False, save_total_limit=2,
    bf16=bf16, fp16=not bf16, report_to='none', seed=42,
    optim='adamw_torch', label_names=['labels']
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=val_ds, data_collator=collate)
start = time.time()
training_result = trainer.train()
elapsed = time.time() - start
model.config.use_cache = True
model.save_pretrained(ROOT / 'adapter')
tok.save_pretrained(ROOT / 'adapter')
(ROOT / 'training-log.json').write_text(json.dumps(trainer.state.log_history, indent=2))
manifest = {
    'base_model': MODEL, 'base_revision': REVISION, 'seed': 42,
    'gpu': torch.cuda.get_device_name(0), 'seconds': elapsed,
    'trainable_parameters': sum(p.numel() for _, p in trainable),
    'total_parameters': sum(p.numel() for p in model.parameters()),
    'split_sizes': {s: len(v) for s, v in data.items()},
    'training_arguments': args.to_dict(), 'training_metrics': training_result.metrics,
    'best_checkpoint': trainer.state.best_model_checkpoint,
    'method': 'LoRA, non-quantized base'
}
(ROOT / 'run.json').write_text(json.dumps(manifest, indent=2, default=str))
(ROOT / 'requirements-lock.txt').write_text(subprocess.check_output(
    [sys.executable, '-m', 'pip', 'freeze'], text=True))
assert (ROOT / 'adapter' / 'adapter_model.safetensors').exists()
print('Training seconds:', elapsed)

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


<IPython.core.display.HTML object>

Training seconds: 96.47100639343262


In [9]:
for name in ("adapter_config.json", "adapter_model.safetensors"):
    path = ROOT / "adapter" / name
    assert path.exists(), f"Missing: {path}"
    print(f"{name}: {path.stat().st_size:,} bytes")

print("Selected checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation loss:", trainer.state.best_metric)

adapter_config.json: 1,080 bytes
adapter_model.safetensors: 2,176,800 bytes
Selected checkpoint: /content/household-lora/checkpoints/checkpoint-20
Best validation loss: 0.723834753036499


In [10]:
adapted_test = save_predictions('test', 'adapted')
def score(predictions):
    c = dict(n=len(data['test']), json_valid=0, schema_valid=0,
             exact_patch=0, clarification_set_correct=0, unrequested_update_errors=0)
    by_id = {p['id']: p['raw'] for p in predictions}
    for r in data['test']:
        try:
            o = json.loads(by_id[r['id']])
            c['json_valid'] += 1
        except (ValueError, TypeError):
            continue
        if not schema_ok(o):
            continue
        c['schema_valid'] += 1
        target = r['target']
        clarify_correct = set(o['clarify']) == set(target['clarify'])
        c['clarification_set_correct'] += int(clarify_correct)
        c['exact_patch'] += int(o['updates'] == target['updates'] and clarify_correct)
        c['unrequested_update_errors'] += int(bool(set(o['updates']) - set(target['updates'])))
    return c

results = {'base': score(base_test), 'adapted': score(adapted_test)}
(ROOT / 'metrics.json').write_text(json.dumps(results, indent=2))
print(json.dumps(results, indent=2))
for r, b, a in zip(data['test'], base_test, adapted_test):
    if b['raw'] != a['raw']:
        print(r['id'], r['input'], '\nEXPECTED', r['target'],
              '\nBASE', b['raw'], '\nADAPTED', a['raw'])
# Also inspect identical-but-wrong outputs; raw equality does not imply correctness.

{
  "base": {
    "n": 24,
    "json_valid": 24,
    "schema_valid": 2,
    "exact_patch": 0,
    "clarification_set_correct": 0,
    "unrequested_update_errors": 0
  },
  "adapted": {
    "n": 24,
    "json_valid": 21,
    "schema_valid": 0,
    "exact_patch": 0,
    "clarification_set_correct": 0,
    "unrequested_update_errors": 0
  }
}
test-001 {'current_state': {'adults': 3, 'budget_eur': 90, 'dairy': 'no_restriction'}, 'instruction': 'My budget for this shop is 47.50 euros.'} 
EXPECTED {'updates': {'budget_eur': 47.5}, 'clarify': []} 
BASE {"updates": {}, "clarify": ["budget_eur"]} 
ADAPTED {"updates": {}, "clarify": ["Dairy should be 'no_restriction'"]}
test-002 {'current_state': {'adults': 2, 'budget_eur': 65, 'dairy': 'dairy_free'}, 'instruction': 'Please plan for seven adults.'} 
EXPECTED {'updates': {'adults': 7}, 'clarify': []} 
BASE {"updates": {}, "clarify": ["set adults to 7"]} 
ADAPTED {"updates": {}, "clarify": ["set adult count to 7"]}
test-003 {'current_state': {'adu

In [11]:
import json
import random
from collections import Counter

# 1. Check the training targets.
bad_targets = [
    r["id"] for r in data["train"]
    if not schema_ok(r["target"])
]

print("Training examples:", len(data["train"]))
print("Invalid training target IDs:", bad_targets)

print(
    "Training clarify types:",
    dict(Counter(
        type(r["target"]["clarify"]).__name__
        for r in data["train"]
    ))
)

print(
    "Training cases with no requested updates:",
    sum(not r["target"]["updates"] for r in data["train"])
)

# 2. Check whether prepared training data matches today's prompt.
# This matters because we revised SYSTEM and restarted the runtime.
assert len(train_ds) == len(data["train"])

mismatches = []

for i, row in enumerate(data["train"]):
    freshly_encoded = encode(row)

    if any(
        train_ds[i][key] != freshly_encoded[key]
        for key in ("input_ids", "attention_mask", "labels")
    ):
        mismatches.append(row["id"])

print("Prepared-data/current-prompt mismatches:", mismatches)

# Show what one example actually asks the model to learn.
sample = train_ds[0]
print("\nFIRST TRAINING INPUT:")
print(data["train"][0]["input"])

print("\nEXPECTED TARGET:")
print(data["train"][0]["target"])

print("\nACTUAL SUPERVISED TOKENS:")
print(tok.decode([
    token for token in sample["labels"]
    if token != -100
]))

# 3. Evaluate a fixed small training sample and all validation cases.
def inspect_cases(rows, name):
    counts = {
        "n": len(rows),
        "json_valid": 0,
        "schema_valid": 0,
        "exact_patch": 0,
    }

    for i, row in enumerate(rows):
        raw = predict(row)

        try:
            parsed = json.loads(raw)
            counts["json_valid"] += 1
        except (ValueError, TypeError):
            parsed = None

        if schema_ok(parsed):
            counts["schema_valid"] += 1
            expected = row["target"]

            correct = (
                parsed["updates"] == expected["updates"]
                and set(parsed["clarify"]) == set(expected["clarify"])
            )

            counts["exact_patch"] += int(correct)

        if i < 3:
            print(f"\n{name}: {row['id']}")
            print("INPUT:", row["input"])
            print("EXPECTED:", row["target"])
            print("RAW:", raw)

    print(f"\n{name} COUNTS:")
    print(json.dumps(counts, indent=2))
    return counts


train_sample = random.Random(42).sample(
    data["train"],
    min(8, len(data["train"]))
)

train_diagnostic = inspect_cases(train_sample, "TRAIN SAMPLE")
val_diagnostic = inspect_cases(data["val"], "VALIDATION")

print("\nOptimizer steps completed:", trainer.state.global_step)
print("Selected checkpoint:", trainer.state.best_model_checkpoint)

Training examples: 80
Invalid training target IDs: []
Training clarify types: {'list': 80}
Training cases with no requested updates: 27
Prepared-data/current-prompt mismatches: []

FIRST TRAINING INPUT:
{'current_state': {'adults': 3, 'budget_eur': 100, 'dairy': 'no_restriction'}, 'instruction': 'Set the budget to 65 euros.'}

EXPECTED TARGET:
{'updates': {'budget_eur': 65}, 'clarify': []}

ACTUAL SUPERVISED TOKENS:
{"updates":{"budget_eur":65},"clarify":[]}<|im_end|>

TRAIN SAMPLE: train-015
INPUT: {'current_state': {'adults': 4, 'budget_eur': 90, 'dairy': 'dairy_free'}, 'instruction': 'Set the budget to 110 euros and remove the dairy restriction.'}
EXPECTED: {'updates': {'budget_eur': 110, 'dairy': 'no_restriction'}, 'clarify': []}
RAW: {"updates": {"budget_eur": 110}, "clarify": {"dairy": "no_restriction"}}

TRAIN SAMPLE: train-004
INPUT: {'current_state': {'adults': 1, 'budget_eur': 40, 'dairy': 'dairy_free'}, 'instruction': 'Bump the budget up to 45 euros please.'}
EXPECTED: {'upd

In [12]:
from datasets import Dataset
from transformers import Trainer, TrainingArguments
import torch

# Three examples we have already inspected, plus one no-change example.
by_id = {row["id"]: row for row in data["train"]}

no_change = next(
    row for row in data["train"]
    if row["target"] == {"updates": {}, "clarify": []}
)

probe_rows = [
    by_id["train-015"],
    by_id["train-004"],
    by_id["train-036"],
    no_change,
]

probe_dataset = Dataset.from_list([
    encode(row) for row in probe_rows
])

# Preserve the original adapter parameters.
parameters = {
    name: parameter
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
}

assert parameters
assert all("lora_" in name for name in parameters)

saved_weights = {
    name: parameter.detach().cpu().clone()
    for name, parameter in parameters.items()
}

previous_cache_setting = model.config.use_cache
probe_trainer = None

try:
    model.config.use_cache = False

    probe_args = TrainingArguments(
        output_dir=str(ROOT / "memorization-diagnostic"),
        max_steps=80,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        learning_rate=1e-4,
        lr_scheduler_type="constant",
        warmup_steps=0,
        logging_steps=20,
        eval_strategy="no",
        save_strategy="no",
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        optim="adamw_torch",
        report_to="none",
        seed=42,
        label_names=["labels"],
    )

    probe_trainer = Trainer(
        model=model,
        args=probe_args,
        train_dataset=probe_dataset,
        data_collator=collate,
    )

    probe_trainer.train()

    largest_change = max(
        (
            parameter.detach().float().cpu()
            - saved_weights[name].float()
        ).abs().max().item()
        for name, parameter in parameters.items()
    )

    print("\nLargest adapter-weight change:", largest_change)

    model.config.use_cache = True

    # Uses the inspection function from your previous diagnostic cell.
    # Prints three examples and scores all four.
    probe_counts = inspect_cases(
        probe_rows,
        "FOUR-EXAMPLE MEMORIZATION CHECK"
    )

finally:
    # Restore the adapter from before this diagnostic.
    with torch.no_grad():
        for name, parameter in parameters.items():
            parameter.copy_(
                saved_weights[name].to(
                    device=parameter.device,
                    dtype=parameter.dtype
                )
            )
            parameter.grad = None

    model.config.use_cache = previous_cache_setting
    model.eval()

    del probe_trainer
    del saved_weights
    torch.cuda.empty_cache()

    print("\nOriginal adapter weights restored.")

<IPython.core.display.HTML object>


Largest adapter-weight change: 0.005286612547934055

FOUR-EXAMPLE MEMORIZATION CHECK: train-015
INPUT: {'current_state': {'adults': 4, 'budget_eur': 90, 'dairy': 'dairy_free'}, 'instruction': 'Set the budget to 110 euros and remove the dairy restriction.'}
EXPECTED: {'updates': {'budget_eur': 110, 'dairy': 'no_restriction'}, 'clarify': []}
RAW: {"updates":{"budget_eur":110,"dairy":"no_restriction"},"clarify":[]}

FOUR-EXAMPLE MEMORIZATION CHECK: train-004
INPUT: {'current_state': {'adults': 1, 'budget_eur': 40, 'dairy': 'dairy_free'}, 'instruction': 'Bump the budget up to 45 euros please.'}
EXPECTED: {'updates': {'budget_eur': 45}, 'clarify': []}
RAW: {"updates":{"budget_eur":45},"clarify":[]}

FOUR-EXAMPLE MEMORIZATION CHECK: train-036
INPUT: {'current_state': {'adults': 5, 'budget_eur': 150, 'dairy': 'dairy_free'}, 'instruction': 'No changes to the budget please, but we now have 7 adults.'}
EXPECTED: {'updates': {'adults': 7}, 'clarify': []}
RAW: {"updates":{"adults":7},"clarify":[]

In [17]:
import json
import time
from transformers import Trainer, TrainingArguments

CONT = ROOT / "continuation-01"

# Prevent accidentally overwriting or repeating this experiment.
if CONT.exists():
    raise RuntimeError(
        "continuation-01 already exists. "
        "Do not rerun blindly; inspect the previous run first."
    )

CONT.mkdir()

assert (ROOT / "adapter" / "adapter_model.safetensors").exists()
assert SYSTEM == (ROOT / "prompt.txt").read_text()

model.config.use_cache = False

continuation_args = TrainingArguments(
    output_dir=str(CONT / "checkpoints"),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=8,
    learning_rate=1e-4,
    lr_scheduler_type="linear",
    warmup_steps=0,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    optim="adamw_torch",
    report_to="none",
    seed=42,
    label_names=["labels"],
)

continuation_trainer = Trainer(
    model=model,
    args=continuation_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate,
)

start = time.time()
continuation_result = continuation_trainer.train()
elapsed = time.time() - start

model.config.use_cache = True
model.eval()

# Save the checkpoint selected by validation loss.
model.save_pretrained(CONT / "adapter")
tok.save_pretrained(CONT / "adapter")

(CONT / "training-log.json").write_text(
    json.dumps(continuation_trainer.state.log_history, indent=2)
)

(CONT / "run.json").write_text(json.dumps({
    "base_model": MODEL,
    "base_revision": REVISION,
    "starting_adapter": str(ROOT / "adapter"),
    "description": (
        "Continuation from the original two-epoch adapter; "
        "new optimizer and learning-rate schedule."
    ),
    "additional_optimizer_steps": continuation_trainer.state.global_step,
    "seconds": elapsed,
    "best_checkpoint": continuation_trainer.state.best_model_checkpoint,
    "best_validation_loss": continuation_trainer.state.best_metric,
    "training_arguments": continuation_args.to_dict(),
}, indent=2, default=str))

print("Additional training seconds:", round(elapsed, 1))
print("Additional optimizer steps:",
      continuation_trainer.state.global_step)
print("Selected checkpoint:",
      continuation_trainer.state.best_model_checkpoint)

<IPython.core.display.HTML object>

Additional training seconds: 362.3
Additional optimizer steps: 80
Selected checkpoint: /content/household-lora/continuation-01/checkpoints/checkpoint-70


In [14]:
import difflib

saved_prompt = (ROOT / "prompt.txt").read_text()

print("CURRENT SYSTEM:\n", SYSTEM)
print("\nSAVED prompt.txt:\n", saved_prompt)

print("\nDIFFERENCES — saved versus current:")
print("\n".join(difflib.unified_diff(
    saved_prompt.splitlines(),
    SYSTEM.splitlines(),
    fromfile="saved prompt.txt",
    tofile="current SYSTEM",
    lineterm=""
)))

print(
    "\nEqual after removing outer whitespace:",
    SYSTEM.strip() == saved_prompt.strip()
)

# Check whether the prepared training sequences use current SYSTEM.
mismatches = []

for i, row in enumerate(data["train"]):
    encoded_now = encode(row)

    if any(
        train_ds[i][key] != encoded_now[key]
        for key in ("input_ids", "attention_mask", "labels")
    ):
        mismatches.append(row["id"])

print("\nTraining-data/current-prompt mismatches:", mismatches)

CURRENT SYSTEM:
 You extract explicit household preference updates from a user instruction.
The input contains current_state and instruction.

Return exactly one JSON object:
{"updates": {}, "clarify": []}

OUTPUT RULES:
- Return JSON only. No Markdown fences, explanation, or extra text.
- Always include both "updates" and "clarify".
- "updates" contains only fields the user explicitly requests to set.
- "clarify" contains only field names whose requested values are unclear
  or unsupported by the rules below.
- Never update and clarify the same field.
- Never clarify a field just because the user did not mention it.

ALLOWED FIELDS:
- adults: integer from 1 to 20.
- budget_eur: number from 1 to 10000, expressed in euros.
- dairy: "dairy_free" or "no_restriction".

INTERPRETATION RULES:
- When a final value is explicit and valid, put it in updates.
- A request to increase or decrease something TO a stated value
  gives an explicit final value. Apply that value.
- A request to increase 

In [15]:
continuation_path = ROOT / "continuation-01"

if continuation_path.exists():
    if any(continuation_path.iterdir()):
        print("Folder contains files; leaving it untouched.")
    else:
        continuation_path.rmdir()
        print("Removed empty continuation folder.")

Removed empty continuation folder.


In [16]:
from datetime import datetime, timezone

prompt_path = ROOT / "prompt.txt"

if prompt_path.read_text() != SYSTEM:
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    backup_path = ROOT / f"prompt-old-{timestamp}.txt"
    backup_path.write_bytes(prompt_path.read_bytes())
    print("Old prompt preserved as:", backup_path.name)

prompt_path.write_text(SYSTEM)

assert prompt_path.read_text() == SYSTEM
print("Saved prompt now matches the current training/inference prompt.")

Old prompt preserved as: prompt-old-20260916T162809107825Z.txt
Saved prompt now matches the current training/inference prompt.


In [18]:
import json

counts = {
    "n": len(data["val"]),
    "json_valid": 0,
    "schema_valid": 0,
    "exact_patch": 0,
}
predictions = []

model.eval()

for row in data["val"]:
    raw = predict(row)
    predictions.append({"id": row["id"], "raw": raw})

    try:
        parsed = json.loads(raw)
        counts["json_valid"] += 1
    except (ValueError, TypeError):
        parsed = None

    correct = False

    if schema_ok(parsed):
        counts["schema_valid"] += 1
        correct = (
            parsed["updates"] == row["target"]["updates"]
            and set(parsed["clarify"]) == set(row["target"]["clarify"])
        )
        counts["exact_patch"] += int(correct)

    if not correct:
        print("\nID:", row["id"])
        print("INPUT:", row["input"])
        print("EXPECTED:", row["target"])
        print("RAW:", raw)

(CONT / "validation-predictions.json").write_text(
    json.dumps(predictions, indent=2)
)
(CONT / "validation-metrics.json").write_text(
    json.dumps(counts, indent=2)
)

print("\nVALIDATION RESULTS:")
print(json.dumps(counts, indent=2))


ID: val-004
INPUT: {'current_state': {'adults': 6, 'budget_eur': 300, 'dairy': 'dairy_free'}, 'instruction': "We're 3 adults now, no dairy restriction, and 90 euros total."}
EXPECTED: {'updates': {'adults': 3, 'dairy': 'no_restriction', 'budget_eur': 90}, 'clarify': []}
RAW: {"updates":{"adults":3,"budget_eur":90},"clarify":[]}

ID: val-008
INPUT: {'current_state': {'adults': 2, 'budget_eur': 60, 'dairy': 'dairy_free'}, 'instruction': "We're not dairy-free anymore, but leave everything else untouched."}
EXPECTED: {'updates': {'dairy': 'no_restriction'}, 'clarify': []}
RAW: {"updates":{"dairy":"dairy_free"},"clarify":[]}

ID: val-009
INPUT: {'current_state': {'adults': 3, 'budget_eur': 90, 'dairy': 'no_restriction'}, 'instruction': "Adults could be 4 or 7, we haven't settled on a number."}
EXPECTED: {'updates': {}, 'clarify': ['adults']}
RAW: {"updates":{"adults":4,"budget_eur":90,"dairy":"no_restriction"},"clarify":[]}

ID: val-010
INPUT: {'current_state': {'adults': 2, 'budget_eur': 

In [19]:
import json
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

adapter_path = CONT / "adapter"

assert (adapter_path / "adapter_model.safetensors").exists()
assert SYSTEM == (ROOT / "prompt.txt").read_text()

# Save the prompt alongside this particular adapter.
(CONT / "prompt.txt").write_text(SYSTEM)

# Use a validation example for this artifact check.
row = data["val"][0]
before = predict(row)

reload_tok = AutoTokenizer.from_pretrained(adapter_path)

reload_base = AutoModelForCausalLM.from_pretrained(
    MODEL,
    revision=REVISION,
    torch_dtype=torch.float32,
).to("cuda")

reload_model = PeftModel.from_pretrained(
    reload_base,
    str(adapter_path),
)
reload_model.eval()

text = reload_tok.apply_chat_template(
    [
        {"role": "system", "content": SYSTEM},
        {
            "role": "user",
            "content": json.dumps(row["input"], ensure_ascii=False),
        },
    ],
    tokenize=False,
    add_generation_prompt=True,
)

inputs = reload_tok(
    text,
    add_special_tokens=False,
    return_tensors="pt",
).to("cuda")

with torch.inference_mode():
    output = reload_model.generate(
        **inputs,
        do_sample=False,
        max_new_tokens=160,
        pad_token_id=reload_tok.pad_token_id,
        eos_token_id=reload_tok.eos_token_id,
    )

after = reload_tok.decode(
    output[0, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
).strip()

result = {
    "example_id": row["id"],
    "before_reload": before,
    "after_reload": after,
    "match": before == after,
}

(CONT / "reload-check.json").write_text(
    json.dumps(result, indent=2)
)

print(json.dumps(result, indent=2))

del reload_model, reload_base, reload_tok, inputs, output
gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/peft_model.py:642: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.2.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.2.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.2.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.2.self_attn.v_proj.l

{
  "example_id": "val-001",
  "before_reload": "{\"updates\":{\"budget_eur\":55},\"clarify\":[]}",
  "after_reload": "{\"updates\": {}, \"clarify\": [\"set budget to 55 euros\"]}",
  "match": false
}


In [20]:
import json
import shutil
from datetime import datetime, timezone
from importlib.metadata import version
from safetensors.torch import save_file

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RECOVERY = ROOT / f"adapter-recovery-{stamp}"
RECOVERY.mkdir()

# Preserve the actual LoRA parameters directly from the working model.
raw_lora = {
    name: parameter.detach().cpu().contiguous().clone()
    for name, parameter in model.named_parameters()
    if "lora_A." in name or "lora_B." in name
}

assert raw_lora, "No LoRA parameters found in the working model."

save_file(raw_lora, str(RECOVERY / "raw-lora-weights.safetensors"))

configs = {
    name: config.to_dict()
    for name, config in model.peft_config.items()
}

packages = {
    name: version(name)
    for name in ("torch", "transformers", "peft", "accelerate", "safetensors")
}

(RECOVERY / "recovery-info.json").write_text(json.dumps({
    "base_model": MODEL,
    "base_revision": REVISION,
    "adapter_configs": configs,
    "packages": packages,
}, indent=2, default=str))

(RECOVERY / "prompt.txt").write_text(SYSTEM)
tok.save_pretrained(RECOVERY / "tokenizer")

print("Recovery snapshot:", RECOVERY)
print("LoRA tensors preserved:", len(raw_lora))

from google.colab import files

archive = shutil.make_archive(str(RECOVERY), "zip", str(RECOVERY))
files.download(archive)

Recovery snapshot: /content/household-lora/adapter-recovery-20260916T164556055524Z
LoRA tensors preserved: 96


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
from safetensors.torch import load_file
from peft import get_peft_model_state_dict

print("PACKAGE VERSIONS:")
print(json.dumps(packages, indent=2))

print("\nMODEL CLASS:", type(model).__name__)
print("ACTIVE ADAPTERS:", model.active_adapters)
print("AVAILABLE ADAPTERS:", list(model.peft_config))

adapter_file = CONT / "adapter" / "adapter_model.safetensors"
disk_state = load_file(str(adapter_file), device="cpu")

print("\nSAVED FILE SIZE:", adapter_file.stat().st_size, "bytes")
print("SAVED TENSOR COUNT:", len(disk_state))

print("\nFIRST SAVED KEYS:")
for key in list(disk_state)[:6]:
    print(key, tuple(disk_state[key].shape))

print("\nFIRST RAW IN-MEMORY KEYS:")
for key in list(raw_lora)[:6]:
    print(key, tuple(raw_lora[key].shape))

print("\nSAVED CONFIG:")
print((CONT / "adapter" / "adapter_config.json").read_text())

# Use PEFT's own export naming rather than guessing a conversion.
exported = get_peft_model_state_dict(
    model,
    adapter_name="default",
)

print("\nPEFT EXPORT TENSOR COUNT:", len(exported))
print("FIRST PEFT EXPORT KEYS:")
for key in list(exported)[:6]:
    print(key, tuple(exported[key].shape))

missing = set(exported) - set(disk_state)
extra = set(disk_state) - set(exported)

print("\nExport keys missing from disk:", len(missing))
print("Examples:", sorted(missing)[:6])
print("Extra disk keys:", len(extra))
print("Examples:", sorted(extra)[:6])

different = [
    key for key in set(exported) & set(disk_state)
    if not torch.equal(
        exported[key].detach().cpu(),
        disk_state[key]
    )
]

print("Shared keys with different tensor values:", len(different))
print("Examples:", sorted(different)[:6])

PACKAGE VERSIONS:
{
  "torch": "2.11.0+cu128",
  "transformers": "5.17.0",
  "peft": "0.21.0",
  "accelerate": "1.15.0",
  "safetensors": "0.8.0"
}

MODEL CLASS: PeftModelForCausalLM
ACTIVE ADAPTERS: ['default']
AVAILABLE ADAPTERS: ['default']

SAVED FILE SIZE: 2176800 bytes
SAVED TENSOR COUNT: 96

FIRST SAVED KEYS:
base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.weight (8, 896)
base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.weight (896, 8)
base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.weight (8, 896)
base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.weight (128, 8)
base_model.model.base_model.model.model.layers.1.self_attn.q_proj.lora_A.weight (8, 896)
base_model.model.base_model.model.model.layers.1.self_attn.q_proj.lora_B.weight (896, 8)

FIRST RAW IN-MEMORY KEYS:
base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight (8, 896)
base_model.model.base_mo

In [22]:
import json
from datetime import datetime, timezone
from safetensors.torch import load_file, save_file

source_dir = CONT / "adapter"
source_weights = load_file(
    str(source_dir / "adapter_model.safetensors"),
    device="cpu",
)

duplicated_prefix = "base_model.model.base_model.model."
one_wrapper = "base_model.model."

# Only apply the exact correction supported by the diagnostics.
assert len(source_weights) == 96
assert all(
    key.startswith(duplicated_prefix)
    for key in source_weights
), "Unexpected key structure. Stop rather than guessing."

corrected_weights = {}

for key, tensor in source_weights.items():
    corrected_key = key[len(one_wrapper):]

    assert corrected_key not in corrected_weights
    corrected_weights[corrected_key] = tensor.contiguous()

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
REPAIRED = CONT / f"adapter-repaired-{stamp}"
REPAIRED.mkdir()

save_file(
    corrected_weights,
    str(REPAIRED / "adapter_model.safetensors"),
    metadata={"format": "pt"},
)

config = json.loads(
    (source_dir / "adapter_config.json").read_text()
)

# Restore the base-model identity missing from the saved config.
config["base_model_name_or_path"] = MODEL
config["revision"] = REVISION

(REPAIRED / "adapter_config.json").write_text(
    json.dumps(config, indent=2)
)

tok.save_pretrained(REPAIRED)
(REPAIRED / "prompt.txt").write_text(SYSTEM)

print("Repaired copy:", REPAIRED)
print("Tensor count:", len(corrected_weights))
print("Example key:", next(iter(corrected_weights)))

Repaired copy: /content/household-lora/continuation-01/adapter-repaired-20260916T164722737296Z
Tensor count: 96
Example key: base_model.model.model.layers.0.self_attn.q_proj.lora_A.weight


In [23]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, get_peft_model_state_dict

fresh_base = AutoModelForCausalLM.from_pretrained(
    MODEL,
    revision=REVISION,
    torch_dtype=torch.float32,
).to("cuda")

assert not any(
    "lora_" in name
    for name, _ in fresh_base.named_parameters()
), "The fresh base unexpectedly already contains adapters."

verified_model = PeftModel.from_pretrained(
    fresh_base,
    str(REPAIRED),
)
verified_model.eval()

# Verify all exported adapter tensors, not just one generated answer.
loaded_weights = get_peft_model_state_dict(
    verified_model,
    adapter_name="default",
)

assert set(loaded_weights) == set(corrected_weights), (
    "Loaded adapter keys do not match the repaired checkpoint."
)

different = [
    key
    for key in corrected_weights
    if not torch.equal(
        loaded_weights[key].detach().cpu(),
        corrected_weights[key],
    )
]

assert not different, f"Tensor mismatch: {different[:5]}"
print("All 96 adapter tensors loaded with exact values.")

# Replay the same validation example using the saved tokenizer.
verified_tok = AutoTokenizer.from_pretrained(REPAIRED)
row = data["val"][0]

before = predict(row)  # Original working model remains unchanged.

prompt = verified_tok.apply_chat_template(
    [
        {"role": "system", "content": SYSTEM},
        {
            "role": "user",
            "content": json.dumps(row["input"], ensure_ascii=False),
        },
    ],
    tokenize=False,
    add_generation_prompt=True,
)

inputs = verified_tok(
    prompt,
    add_special_tokens=False,
    return_tensors="pt",
).to("cuda")

with torch.inference_mode():
    output = verified_model.generate(
        **inputs,
        do_sample=False,
        max_new_tokens=160,
        pad_token_id=verified_tok.pad_token_id,
        eos_token_id=verified_tok.eos_token_id,
    )

after = verified_tok.decode(
    output[0, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
).strip()

check = {
    "adapter_tensors_match": True,
    "example_id": row["id"],
    "before_reload": before,
    "after_reload": after,
    "prediction_match": before == after,
}

(REPAIRED / "reload-check.json").write_text(
    json.dumps(check, indent=2)
)

print(json.dumps(check, indent=2))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

All 96 adapter tensors loaded with exact values.
{
  "adapter_tensors_match": true,
  "example_id": "val-001",
  "before_reload": "{\"updates\":{\"budget_eur\":55},\"clarify\":[]}",
  "after_reload": "{\"updates\":{\"budget_eur\":55},\"clarify\":[]}",
  "prediction_match": true
}


In [24]:
import json
import torch

saved_predictions = {
    item["id"]: item["raw"]
    for item in json.loads(
        (CONT / "validation-predictions.json").read_text()
    )
}

counts = {
    "n": len(data["val"]),
    "json_valid": 0,
    "schema_valid": 0,
    "exact_patch": 0,
    "matches_saved_prediction": 0,
}
replayed = []

verified_model.eval()

for row in data["val"]:
    prompt = verified_tok.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM},
            {
                "role": "user",
                "content": json.dumps(row["input"], ensure_ascii=False),
            },
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = verified_tok(
        prompt,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")

    with torch.inference_mode():
        output = verified_model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=160,
            pad_token_id=verified_tok.pad_token_id,
            eos_token_id=verified_tok.eos_token_id,
        )

    raw = verified_tok.decode(
        output[0, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    replayed.append({"id": row["id"], "raw": raw})

    matches = raw == saved_predictions[row["id"]]
    counts["matches_saved_prediction"] += int(matches)

    if not matches:
        print("Replay differs:", row["id"])
        print("SAVED:", saved_predictions[row["id"]])
        print("RELOADED:", raw)

    try:
        parsed = json.loads(raw)
        counts["json_valid"] += 1
    except (ValueError, TypeError):
        parsed = None

    if schema_ok(parsed):
        counts["schema_valid"] += 1
        counts["exact_patch"] += int(
            parsed["updates"] == row["target"]["updates"]
            and set(parsed["clarify"]) == set(row["target"]["clarify"])
        )

(REPAIRED / "validation-replay.json").write_text(
    json.dumps(replayed, indent=2)
)
(REPAIRED / "validation-replay-metrics.json").write_text(
    json.dumps(counts, indent=2)
)

print(json.dumps(counts, indent=2))

{
  "n": 16,
  "json_valid": 16,
  "schema_valid": 15,
  "exact_patch": 10,
  "matches_saved_prediction": 16
}


In [25]:
import shutil
from google.colab import files

archive = shutil.make_archive(
    str(REPAIRED),
    "zip",
    str(REPAIRED),
)

files.download(archive)
print("Downloaded repaired adapter:", REPAIRED.name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded repaired adapter: adapter-repaired-20260916T164722737296Z


In [26]:
import json
import hashlib
from contextlib import nullcontext
from google.colab import files
import torch

# Upload only final_test.jsonl.
# Use returned bytes directly, so Colab's filename suffixes don't matter.
uploaded_final = files.upload()
assert len(uploaded_final) == 1, "Upload only final_test.jsonl."

payload = next(iter(uploaded_final.values()))

final_rows = [
    json.loads(line)
    for line in payload.decode("utf-8").splitlines()
    if line.strip()
]

assert len(final_rows) == 24
assert len({r["id"] for r in final_rows}) == 24
assert all(schema_ok(r["target"]) for r in final_rows)

# Use exactly the prompt saved with the repaired adapter.
FINAL_SYSTEM = (REPAIRED / "prompt.txt").read_text()
assert SYSTEM == FINAL_SYSTEM

verified_model.eval()
assert verified_model.active_adapters == ["default"]

# The adapter must be unmerged for disabling it to recover the base.
assert all(
    not module.merged_adapters
    for module in verified_model.modules()
    if hasattr(module, "merged_adapters")
)

OUT = CONT / "final-evaluation"

if OUT.exists():
    raise RuntimeError(
        "final-evaluation already exists. Inspect its results "
        "before running this evaluation again."
    )

OUT.mkdir()
(OUT / "final_test.jsonl").write_bytes(payload)
(OUT / "prompt.txt").write_text(FINAL_SYSTEM)


def final_predict(row):
    prompt = verified_tok.apply_chat_template(
        [
            {"role": "system", "content": FINAL_SYSTEM},
            {
                "role": "user",
                "content": json.dumps(
                    row["input"], ensure_ascii=False
                ),
            },
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = verified_tok(
        prompt,
        add_special_tokens=False,
        return_tensors="pt",
    ).to(next(verified_model.parameters()).device)

    with torch.inference_mode():
        output = verified_model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=160,
            pad_token_id=verified_tok.pad_token_id,
            eos_token_id=verified_tok.eos_token_id,
        )

    return verified_tok.decode(
        output[0, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()


results = {}

for tag in ("base", "adapted"):
    counts = {
        "n": len(final_rows),
        "json_valid": 0,
        "schema_valid": 0,
        "exact_patch": 0,
    }
    predictions = []

    context = (
        verified_model.disable_adapter()
        if tag == "base"
        else nullcontext()
    )

    with context:
        for row in final_rows:
            raw = final_predict(row)
            predictions.append({"id": row["id"], "raw": raw})

            try:
                parsed = json.loads(raw)
                counts["json_valid"] += 1
            except (ValueError, TypeError):
                parsed = None

            if schema_ok(parsed):
                counts["schema_valid"] += 1
                counts["exact_patch"] += int(
                    parsed["updates"] == row["target"]["updates"]
                    and set(parsed["clarify"])
                    == set(row["target"]["clarify"])
                )

    results[tag] = counts

    (OUT / f"{tag}-predictions.json").write_text(
        json.dumps(predictions, indent=2)
    )
    print(f"{tag}: completed {len(predictions)} cases")

(OUT / "metrics.json").write_text(
    json.dumps(results, indent=2)
)

(OUT / "evaluation-config.json").write_text(json.dumps({
    "base_model": MODEL,
    "base_revision": REVISION,
    "adapter_path": str(REPAIRED),
    "dataset_sha256": hashlib.sha256(payload).hexdigest(),
    "prompt_sha256": hashlib.sha256(
        FINAL_SYSTEM.encode("utf-8")
    ).hexdigest(),
    "do_sample": False,
    "max_new_tokens": 160,
    "baseline_method": "Repaired model with adapter disabled",
}, indent=2))

print("\nFINAL EVALUATION:")
print(json.dumps(results, indent=2))

<IPython.core.display.HTML object>

Saving final_test.jsonl to final_test.jsonl
base: completed 24 cases
adapted: completed 24 cases

FINAL EVALUATION:
{
  "base": {
    "n": 24,
    "json_valid": 22,
    "schema_valid": 2,
    "exact_patch": 0
  },
  "adapted": {
    "n": 24,
    "json_valid": 24,
    "schema_valid": 19,
    "exact_patch": 12
  }
}


In [27]:
import zipfile
from google.colab import files

bundle = ROOT.parent / "household-lora-completed-experiment.zip"

with zipfile.ZipFile(
    bundle, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in ROOT.rglob("*"):
        if (
            path.is_file()
            and "checkpoints" not in path.relative_to(ROOT).parts
            and path.suffix != ".zip"
        ):
            archive.write(path, path.relative_to(ROOT))

files.download(str(bundle))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>